# LAID baseline 1: Community Forensics 224

This is a **measuring stick, not the final LAID detector**. It runs the official MIT-licensed Community Forensics 224 weights at the bounty's fixed `0.65` threshold, then applies LAID's JPEG and resize stress tests.

Before running: enable a GPU and Internet, then click **Add Input** and attach `rhythmghai/ai-vs-real-images-dataset`. The notebook samples 100 real and 100 SDXL images; it does not copy the full dataset into working storage.

In [ ]:
import shutil
import subprocess
from pathlib import Path
import torch

if not torch.cuda.is_available():
    raise RuntimeError('No GPU attached. Enable a GPU in Session options and reconnect.')
dataset = Path('/kaggle/input/ai-vs-real-images-dataset')
if not dataset.exists():
    raise RuntimeError('Dataset missing. Click Add Input and attach rhythmghai/ai-vs-real-images-dataset.')
disk = shutil.disk_usage('/kaggle/working')
if disk.free < 10 * 1024**3:
    raise RuntimeError(f'Less than 10 GB free: {disk.free / 1024**3:.1f} GB')
print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'Free disk: {disk.free / 1024**3:.1f} GB')


In [ ]:
laid = Path('/kaggle/working/laid')
if (laid / '.git').exists():
    subprocess.run(['git', '-C', str(laid), 'pull', '--ff-only'], check=True)
else:
    subprocess.run(['git', 'clone', 'https://github.com/Eienel/laid.git', str(laid)], check=True)
subprocess.run(['python', '-m', 'pip', 'install', '-q', '-e', str(laid), 'timm==1.0.15', 'huggingface-hub'], check=True)

official = Path('/kaggle/working/community-forensics')
commit = 'ee5b71d43db0f3779e1edd64ee927b13f2dd6ad4'
if not (official / '.git').exists():
    subprocess.run(['git', 'clone', 'https://github.com/JeongsooP/Community-Forensics.git', str(official)], check=True)
subprocess.run(['git', '-C', str(official), 'checkout', '--detach', commit], check=True)


In [ ]:
subprocess.run([
    'python', str(laid / 'scripts' / 'kaggle_commfor_baseline.py'),
    '--input-root', str(dataset), '--per-class', '100', '--batch-size', '32'
], check=True)


In [ ]:
import json
results = laid / 'results' / 'community-forensics-224'
report = json.loads((results / 'report.json').read_text())
provenance = json.loads((results / 'provenance.json').read_text())
print('Overall balanced accuracy:', f"{report['overall']['balanced_accuracy']:.3f}")
for name, values in report['by_degradation'].items():
    print(name, f"{values['balanced_accuracy']:.3f}")
print('Mean GPU inference ms/image:', provenance['mean_inference_ms_per_image'])
print('Artifacts:', results)


## What to send back

Send a screenshot of the last cell. The meaningful outputs are clean/JPEG/downscale balanced accuracy at `0.65` and GPU milliseconds per image. Stop the session afterward.